# 58 · Ensemble epoch trong **cùng** lần chạy — 0 giây GPU

## Vì sao làm cái này trước

`57` cho thấy `tail` của một epoch đơn dao động **±0.03** giữa các epoch, và **hai seed cho hai quỹ
đạo ngược nhau**:

```
tpb tail  s42  .0785 .1100 .1004 .0984 .0999 .0986 .0986 .0979 .0983 .0972
          s43  .0861 .0946 .1002 .1019 .1078 .0988
```

Không có dữ liệu nào chọn được `E`. Và quan trọng hơn: **không thể đo một hiệu ứng `+0.005` trên
một cái thước lắc `±0.03`**. Mọi thí nghiệm long-tail sau đây (hướng #2, `nb55`) đều so vào mốc
này, nên phải làm mốc đứng yên trước.

Ensemble không phải để cộng điểm — nó **co nhiễu của chính cái mốc**.

## Không tốn gì

`53` và `57` đã lưu dự đoán val (`p_*`) và lát chọn (`psl_*`) của **từng** epoch. Cell này chỉ đọc
và trung bình chúng. Không GPU, không huấn luyện lại.

## Kỷ luật

**Chọn trên lát chọn, báo cáo trên val.** "Epoch đơn tốt nhất" được chọn bằng `psl_*` chứ không bằng
`p_*` — chọn trên val rồi báo cáo trên val là đúng cái bẫy `§13`. Cửa sổ ensemble cũng vậy.

**Trung bình HẠNG song song trung bình xác suất.** AP là chỉ số **xếp hạng**, và các epoch/seed lệch
calibration; `48` và `17` đều đo cả hai.

**Bootstrap theo BỆNH NHÂN.** Hai phim cùng người không độc lập; lấy mẫu theo ảnh cho khoảng **hẹp
giả**.

## Mốc đối chiếu

`17` đo trên cohort cũ (13 nhãn, **1** nhãn tail), ensemble 3 checkpoint cùng lần chạy:

```
macro  +0.00112  CI [+0.00027, +0.00197]  loai 0
worst3 +0.00345  CI [+0.00035, +0.00645]  loai 0
tail   +0.00465  CI [-0.00230, +0.01213]  chua 0
```

Mức tăng lớn nhất nằm đúng ở tail. Cần đo lại trên **11** nhãn tail của CXR-LT.

## C0 · Config + nạp nhãn

In [ ]:
import os, json, re, glob
import numpy as np, pandas as pd
try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = os.environ.get('CXRLT_DATA', '')
    assert DATA_DIR, 'khong phai Colab va chua dat CXRLT_DATA -> gan thang duong dan `planB_out`'
OUT_DIR = os.path.join(DATA_DIR, 'cxrlt_2024'); EMB_DIR = os.path.join(OUT_DIR, 'embeddings')
RES58 = os.path.join(OUT_DIR, 'stage4_nb58')
def ensure_res():
    os.makedirs(RES58, exist_ok=True); return RES58
ensure_res()
assert os.path.isdir(OUT_DIR) and os.listdir(OUT_DIR), f'{OUT_DIR} khong doc duoc -> mount hong'

# --- NGUON: doi hai dong nay de doc lan chay khac ---
SRC = {'53': os.path.join(OUT_DIR, 'stage4_nb53', 'r3'),
       '57': os.path.join(OUT_DIR, 'stage4_nb57', 'r3')}
SEED = 42

INV = json.load(open(os.path.join(OUT_DIR, 'cxrlt_inventory.json')))
LAB40 = INV['labels_task1']; C = len(LAB40)
TIER = {t: np.array([LAB40.index(l) for l in INV['tier'][t]]) for t in ('head','medium','tail')}
ORDER = pd.read_parquet(os.path.join(EMB_DIR, 'emb_index.parquet'))
YM = (pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_index.parquet'), columns=['dicom_id'] + LAB40)
        .set_index('dicom_id').loc[ORDER.dicom_id].reset_index())[LAB40].to_numpy('float32')
SPLIT = ORDER.split.to_numpy(); SUB = ORDER.subject_id.to_numpy()
TR, VA = SPLIT == 'train', SPLIT == 'val'
_rs = np.random.default_rng(SEED); _su = np.unique(SUB[TR])
_sel = _rs.choice(_su, size=int(round(0.15*len(_su))), replace=False)
SL = TR & np.isin(SUB, _sel)                       # CUNG cach cat nhu 53/56/57 -> `psl_*` khop hang
Y_VA, Y_SL = YM[VA], YM[SL]
SUB_VA = SUB[VA]
print(f'val {int(VA.sum()):,} anh | lat chon {int(SL.sum()):,} anh | {C} nhan')
for k, d in SRC.items():
    print(f'  {k}: {d}  ->  {"CO" if os.path.isdir(d) else "KHONG THAY"}')

## C1 · Chấm điểm — đối chiếu sklearn

In [ ]:
from sklearn.metrics import average_precision_score
def ap_cols(Yc, Sc):
    o = np.full(Yc.shape[1], np.nan)
    for j in range(Yc.shape[1]):
        y, s = Yc[:, j], Sc[:, j]; n1 = y.sum()
        if n1 == 0 or n1 == len(y): continue
        r = np.argsort(-s, kind='stable'); yy, ss = y[r], s[r]
        tp, fp = np.cumsum(yy), np.cumsum(1 - yy)
        last = np.r_[np.diff(ss) != 0, True]; tp, fp = tp[last], fp[last]
        pr, rc = tp / (tp + fp), tp / n1
        o[j] = float((np.diff(np.r_[0.0, rc]) * pr).sum())
    return o
mac = lambda Y, P: float(np.nanmean(ap_cols(Y, P)))
tie = lambda Y, P: {t: float(np.nanmean(ap_cols(Y, P)[i])) for t, i in TIER.items()}
_r6 = np.random.default_rng(6); _y6 = (_r6.random((3000, C)) < 0.2).astype('float32')
_s6 = _r6.random((3000, C)).astype('float32')
assert np.nanmax(np.abs(ap_cols(_y6, _s6) - np.array(
    [average_precision_score(_y6[:, j], _s6[:, j]) for j in range(C)]))) < 1e-6
print('cong 1: ap_cols khop sklearn')

from scipy.stats import rankdata
def rk(P):
    '''Trung binh HANG: AP la chi so XEP HANG, va cac epoch lech calibration nen trung binh xac
       suat co the bi mot epoch tu tin thai qua keo di.'''
    return rankdata(P, axis=0) / len(P)

## C2 · Nạp dự đoán từng epoch

Đọc `p_{arm}_s{seed}_ep{e}.npy` (val) và `psl_*` (lát chọn). Dừng ồn ào nếu không thấy, kèm danh
sách thứ thật sự có trên đĩa — bài học `41c`: một thư mục rỗng làm mọi thứ phía sau báo sai nguyên
nhân.

In [ ]:
def load_run(dirp, tag):
    if not os.path.isdir(dirp): print(f'  {tag}: KHONG THAY {dirp}'); return {}
    R = {}
    for f in sorted(glob.glob(os.path.join(dirp, 'p_*_s*_ep*.npy'))):
        m = re.match(r'p_(.+)_s(\d+)_ep(\d+)\.npy$', os.path.basename(f))
        if not m: continue
        arm, sd, ep = m.group(1), int(m.group(2)), int(m.group(3))
        fs = os.path.join(dirp, f'psl_{arm}_s{sd}_ep{ep}.npy')
        if not os.path.exists(fs): continue          # can CA HAI: chon tren lat chon, cham tren val
        R.setdefault((tag, arm, sd), {})[ep] = (f, fs)
    return R

RUNS = {}
for k, d in SRC.items(): RUNS.update(load_run(d, k))
assert RUNS, ('khong nap duoc cap (p_*, psl_*) nao. Kiem hai duong dan o C0 — thu muc co the dung '
              'RUN_TAG khac (r1/r2/r3).')
print(f'{"nguon/arm/seed":<34}{"so epoch":>9}  epoch')
for k in sorted(RUNS):
    eps = sorted(RUNS[k])
    print(f'  {k[0]}/{k[1]:<24} s{k[2]} {len(eps):>7}  {eps}')
_va_n, _sl_n = int(VA.sum()), int(SL.sum())
for k, v in RUNS.items():
    _e0 = sorted(v)[0]
    _a = np.load(v[_e0][0]); _b = np.load(v[_e0][1])
    assert _a.shape == (_va_n, C), f'{k} p_* hinh dang {_a.shape}, cho doi {(_va_n, C)}'
    assert _b.shape == (_sl_n, C), f'{k} psl_* hinh dang {_b.shape}, cho doi {(_sl_n, C)}'
print('cong 2: hinh dang khop val/lat chon OK')

## C3 · Đường cong **từng tier** — có bằng phẳng không, có cùng đỉnh không

Câu hỏi của người dùng: *"kiến trúc chưa đảm bảo head / medium / tail tăng đều nhau"*. Bảng này
chỉ ra ba tier đạt đỉnh ở **epoch nào**, trên **lát chọn** (nơi được phép chọn).

In [ ]:
CUR = {}
for k, v in sorted(RUNS.items()):
    eps = sorted(v)
    rows = []
    for e in eps:
        pv, ps = np.load(v[e][0]), np.load(v[e][1])
        rows.append(dict(ep=e, sel=mac(Y_SL, ps), val=mac(Y_VA, pv), **tie(Y_VA, pv)))
    CUR[k] = pd.DataFrame(rows).set_index('ep')
    d = CUR[k]
    print(f'\n=== {k[0]}/{k[1]} s{k[2]} ===')
    print(d.to_string(float_format=lambda x: f'{x:.4f}'))
    pk = {c: int(d[c].idxmax()) for c in ('sel','val','head','medium','tail')}
    print(f'  dinh:  ' + '  '.join(f'{c} ep{pk[c]}' for c in pk)
          + ('   <- BA TIER LECH DINH' if len({pk["head"], pk["medium"], pk["tail"]}) > 1 else ''))

## C4 · Epoch đơn (chọn trên **lát chọn**) vs trung bình cửa sổ

`single` = epoch có `sel` cao nhất, rồi báo cáo `val` của nó — **không** chọn bằng `val`.
`lastK`/`topK` = trung bình dự đoán, cả **xác suất** lẫn **hạng**.

In [ ]:
def combos(v):
    eps = sorted(v); out = {}
    sel = {e: mac(Y_SL, np.load(v[e][1])) for e in eps}
    best = max(sel, key=sel.get)
    out['single(sel)'] = [best]
    for K in (3, 5):
        if len(eps) >= K:
            out[f'last{K}'] = eps[-K:]
            out[f'top{K}(sel)'] = sorted(sorted(eps, key=lambda e: -sel[e])[:K])
    out['all'] = eps
    return out, best, sel

ROWS = []
for k, v in sorted(RUNS.items()):
    cb, best, sel = combos(v)
    for name, eps in cb.items():
        P = [np.load(v[e][0]) for e in eps]
        pm = np.mean(P, 0)
        rm = np.mean([rk(p) for p in P], 0) if len(P) > 1 else pm
        # `sel` cua CHINH to hop nay: ensemble `psl_*` y het cach ensemble `p_*`.
        # Thieu cot nay thi KHONG co cach hop le nao chon cua so — chon bang cot `prob`/`rank`
        # (deu tren val) roi bao cao tren val la dung bay §13.
        sm = np.mean([np.load(v[e][1]) for e in eps], 0)
        ROWS.append(dict(nguon=k[0], arm=k[1], seed=k[2], to_hop=name, n_ep=len(eps),
                         sel=mac(Y_SL, sm),
                         prob=mac(Y_VA, pm), rank=mac(Y_VA, rm),
                         prob_tail=tie(Y_VA, pm)['tail'], rank_tail=tie(Y_VA, rm)['tail']))
T = pd.DataFrame(ROWS)
def best_win(k):
    '''Cua so tot nhat theo LAT CHON. C5/C6 goi ham nay thay vi cho cung `last3`.'''
    d = T[(T.nguon == k[0]) & (T.arm == k[1]) & (T.seed == k[2])].set_index('to_hop')
    return str(d['sel'].idxmax())
for k in sorted(RUNS):
    d = T[(T.nguon == k[0]) & (T.arm == k[1]) & (T.seed == k[2])].set_index('to_hop')
    b = d.loc['single(sel)']
    d = d.assign(d_prob=d['prob'] - b['prob'], d_tail=d['prob_tail'] - b['prob_tail'])
    print(f'\n=== {k[0]}/{k[1]} s{k[2]}  (moc = single chon tren lat chon) ===')
    print(d[['n_ep','sel','prob','rank','prob_tail','rank_tail','d_prob','d_tail']]
          .to_string(float_format=lambda x: f'{x:+.4f}' if abs(x) < 0.02 else f'{x:.4f}'))
    print(f'  -> cua so chon theo LAT CHON: `{d["sel"].idxmax()}`'
          f'   (theo val se la `{d["prob"].idxmax()}` — KHONG duoc dung de chon)')
ensure_res(); T.to_csv(os.path.join(RES58, 'epoch_ensemble.csv'), index=False)

## C5 · Khoảng tin cậy — bootstrap theo **bệnh nhân**

Hiệu số nhỏ thì phải có CI mới đọc được. Lấy mẫu theo **bệnh nhân** (hai phim cùng người không độc
lập; lấy mẫu theo ảnh cho khoảng **hẹp giả**).

In [ ]:
B = 1000
UQ = np.unique(SUB_VA); IDX_OF = {u: np.where(SUB_VA == u)[0] for u in UQ}
def boot(pa, pb, y=Y_VA, cols=None, seed=4):
    r = np.random.default_rng(seed); d = []
    for _ in range(B):
        pick = r.choice(UQ, size=len(UQ), replace=True)
        ix = np.concatenate([IDX_OF[u] for u in pick])
        aa, bb = ap_cols(y[ix], pa[ix]), ap_cols(y[ix], pb[ix])
        if cols is not None: aa, bb = aa[cols], bb[cols]
        d.append(float(np.nanmean(bb)) - float(np.nanmean(aa)))
    d = np.array(d); return float(d.mean()), *np.percentile(d, [2.5, 97.5])

# Bootstrap la phan CHAM nhat (B x 2 x so run). Mac dinh chi arm CHINH — do them arm KHONG
# lam CI hep lai, chi ton thoi gian.
ONLY = ('t1_prior_both', 'tpb')
print(f'{"nguon/arm/seed":<26}{"cua so":>11}{"do":>8}{"delta":>9}{"CI95":>22}')
OUT = []
for k, v in sorted(RUNS.items()):
    if ONLY and k[1] not in ONLY: continue
    cb, best, _ = combos(v)
    WIN = best_win(k)                      # chon bang LAT CHON, khong cho cung `last3`
    if WIN not in cb: continue
    base = np.load(v[best][0]); ens = np.mean([np.load(v[e][0]) for e in cb[WIN]], 0)
    for nm, cols in (('macro', None), ('tail', TIER['tail'])):
        m, lo, hi = boot(base, ens, cols=cols)
        ok = 'LOAI 0' if lo*hi > 0 else 'chua 0'
        print(f'  {k[0]}/{k[1]:<16} s{k[2]}{WIN:>11} {nm:>7}{m:>+9.4f}  [{lo:+.4f}, {hi:+.4f}] {ok}')
        OUT.append(dict(nguon=k[0], arm=k[1], seed=k[2], cua_so=WIN, do=nm,
                        delta=m, lo=lo, hi=hi))
print(f'\nmoc `17` (cohort cu, 13 nhan / 1 nhan tail):')
print('  macro +0.00112 [+0.00027, +0.00197] | tail +0.00465 [-0.00230, +0.01213]')
print('\n=== DOC THE NAO ===')
print('  tail delta > 0 va CI loai 0  -> BO HAN viec chon E; moi thi nghiem long-tail lay')
print('     trung binh cua so lam MOC. Hieu ung cua #2 se do duoc.')
print('  CI chua 0 va delta ~ 0       -> dao dong giu' \
'a cac epoch la NHIEU THUAN, khong khai thac')
print('     duoc. Van nen lay trung binh cua so lam moc vi no BOT NHIEU cho phep do sau.')
ensure_res(); pd.DataFrame(OUT).to_csv(os.path.join(RES58, 'bootstrap.csv'), index=False)

## C6 · Ensemble **seed** chồng lên (R1) — cũng miễn phí

In [ ]:
GR = {}
for (tag, arm, sd), v in RUNS.items(): GR.setdefault((tag, arm), []).append((sd, v))
print(f'{"nguon/arm":<26}{"1 seed":>9}{"2 seed":>9}{"delta":>9}{"tail 1s":>9}{"tail 2s":>9}')
for (tag, arm), lst in sorted(GR.items()):
    if len(lst) < 2: continue
    per, wins = [], []
    for sd, v in lst:
        cb, best, _ = combos(v)
        w = best_win((tag, arm, sd)); wins.append(w)    # chon bang LAT CHON cho TUNG seed
        eps = cb.get(w, [best])
        per.append(np.mean([np.load(v[e][0]) for e in eps], 0))
    one = float(np.mean([mac(Y_VA, p) for p in per]))          # TB cua tung seed
    ens = mac(Y_VA, np.mean(per, 0))                           # ensemble hai seed
    t1 = float(np.mean([tie(Y_VA, p)['tail'] for p in per]))
    t2 = tie(Y_VA, np.mean(per, 0))['tail']
    print(f'  {tag}/{arm:<20}{one:>9.4f}{ens:>9.4f}{ens-one:>+9.4f}{t1:>9.4f}{t2:>9.4f}'
          + '   [' + ','.join(wins) + ']')
print('\n  `48` do ensemble seed +0.0103. Cong don voi ensemble epoch o C4/C5 la khoan RE NHAT')
print('  con lai trong ke hoach — khong ton mot giay GPU nao ngoai gio da chay.')